<a href="https://colab.research.google.com/github/samreenfathima18/Guvi-Final-project/blob/main/04_%E2%80%93_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
import os

from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

**Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load Dataset**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/outputs/"

df = pd.read_parquet(DATA_PATH + "processed_CA1.parquet")

print(df.shape)
df.head()

(5832737, 22)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,None,None,None,None,0,0,0,0.0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,None,None,None,None,0,0,0,0.0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,None,None,None,None,0,0,0,0.0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,None,None,None,None,0,0,0,0.0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,None,None,None,None,0,0,0,0.0


**Sort Data**

In [ ]:
df = (
    df.sort_values(
        ["store_id", "item_id", "date"]
    )
    .reset_index(drop=True)
)

**Section 1 – Time Features**

In [ ]:
df["day"] = df["date"].dt.day

df["week"] = df["date"].dt.isocalendar().week.astype("int16")

df["quarter"] = df["date"].dt.quarter.astype("int8")

df["is_weekend"] = (
    df["weekday"]
    .isin(["Saturday", "Sunday"])
    .astype("int8")
)

df["is_month_start"] = (
    df["date"]
    .dt.is_month_start
    .astype("int8")
)

df["is_month_end"] = (
    df["date"]
    .dt.is_month_end
    .astype("int8")
)

**Section 2 – Lag Features**

In [ ]:
lag_days = [1, 7, 14, 28]

for lag in lag_days:

    df[f"lag_{lag}"] = (
        df.groupby(
            ["store_id", "item_id"]
        )["sales"]
        .shift(lag)
    )

**Section 3 – Rolling Features**

In [ ]:
windows = [7, 14, 28]

for window in windows:

    df[f"rolling_mean_{window}"] = (
        df.groupby(
            ["store_id", "item_id"]
        )["sales"]
        .transform(
            lambda x:
            x.shift(1).rolling(window).mean()
        )
    )

    df[f"rolling_std_{window}"] = (
        df.groupby(
            ["store_id", "item_id"]
        )["sales"]
        .transform(
            lambda x:
            x.shift(1).rolling(window).std()
        )
    )

**Section 4 – Price Features**

In [ ]:
df["price_change"] = (
    df.groupby(
        ["store_id", "item_id"]
    )["sell_price"]
    .pct_change()
)

df["price_diff"] = (
    df.groupby(
        ["store_id", "item_id"]
    )["sell_price"]
    .diff()
)

df["price_rolling_mean"] = (
    df.groupby(
        ["store_id", "item_id"]
    )["sell_price"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

**Price Momentum**

**Section 5 – Event Features**

In [ ]:
df["is_event"] = (
    df["event_name_1"] != "None"
).astype("int8")

df["is_snap"] = (
    df["snap_CA"] == 1
).astype("int8")

**Section 6 – Encode Categorical Variables**

In [ ]:
encoders = {}

categorical_columns = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday"
]

os.makedirs(
    "/content/drive/MyDrive/Enterprise_Retail_Intelligence/models/",
    exist_ok=True
)

for col in categorical_columns:

    le = LabelEncoder()

    df[col] = le.fit_transform(df[col])

    encoders[col] = le

    joblib.dump(
        le,
        f"/content/drive/MyDrive/Enterprise_Retail_Intelligence/models/{col}_encoder.pkl"
    )

**Section 7 – Clean Missing & Infinite Values **

In [ ]:
df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df.fillna(0, inplace=True)

**Verify Data Quality**

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

print("Infinity Values :", np.isinf(numeric_df).sum().sum())

print("Missing Values  :", numeric_df.isnull().sum().sum())

Infinity Values : 0
Missing Values  : 0


**Memory Optimization**

In [ ]:
for col in df.select_dtypes(include="float64"):

    df[col] = pd.to_numeric(
        df[col],
        downcast="float"
    )

for col in df.select_dtypes(include="int64"):

    df[col] = pd.to_numeric(
        df[col],
        downcast="integer"
    )

**Section 8 – Final Dataset**

In [ ]:
print(df.shape)

df.info(memory_usage="deep")

(5832737, 43)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5832737 entries, 0 to 5832736
Data columns (total 43 columns):
 #   Column              Dtype         
---  ------              -----         
 0   id                  object        
 1   item_id             int16         
 2   dept_id             int8          
 3   cat_id              int8          
 4   store_id            int8          
 5   state_id            int8          
 6   d                   object        
 7   sales               int16         
 8   date                datetime64[ns]
 9   wm_yr_wk            int16         
 10  weekday             int8          
 11  wday                int8          
 12  month               int8          
 13  year                int16         
 14  event_name_1        object        
 15  event_type_1        object        
 16  event_name_2        object        
 17  event_type_2        object        
 18  snap_CA             int8          
 19  snap_TX             int8    

**Save Feature Engineered Dataset**

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/outputs/"

df.to_parquet(
    OUTPUT_PATH + "feature_engineered_CA1.parquet",
    index=False
)

print("Feature Engineered Dataset Saved Successfully!")

Feature Engineered Dataset Saved Successfully!


**Business Validation**

In [ ]:
feature_summary = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum().values
})

feature_summary.head(20)

,Feature,Data Type,Missing Values
id,id,object,0
item_id,item_id,int16,0
dept_id,dept_id,int8,0
cat_id,cat_id,int8,0
store_id,store_id,int8,0
state_id,state_id,int8,0
d,d,object,0
sales,sales,int16,0
date,date,datetime64[ns],0
wm_yr_wk,wm_yr_wk,int16,0
